In [2]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
video_path = r"D:\Duy Toan\Python\DUT AI Club\Homework\day 28 3-3-2026\dataset.mp4"

threshold_val = 30

In [ ]:
def background_subtraction(video_path, threshold_val=30):
    cap = cv2.VideoCapture(video_path)

    ret, prev_frame = cap.read()
    if not ret:
        print("Không thể kết nối camera/video.")
        return

    # Làm mịn nhẹ để giảm nhiễu hạt
    prev_frame = cv2.GaussianBlur(prev_frame, (5, 5), 0)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        # làm mịn nhẹ
        current_frame_blur = cv2.GaussianBlur(frame, (5, 5), 0)
        
        frame_diff = cv2.absdiff(current_frame_blur, prev_frame)
        diff_gray = cv2.cvtColor(frame_diff, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(diff_gray, threshold_val, 255, cv2.THRESH_BINARY)
        
        kernel = np.ones((5, 5), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=3)        # Làm dày vật thể
        mask = cv2.erode(mask, kernel, iterations=1)

        # Tìm và vẽ Bounding Box (Khung chữ nhật) lên ảnh màu gốc
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < 500:
                continue
            (x, y, w, h) = cv2.boundingRect(contour)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        cv2.imshow('Object Detection (Color Diff)', frame)
        prev_frame = current_frame_blur
        if cv2.waitKey(30) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

background_subtraction(video_path, threshold_val)